# NB-04: Numerical Consistency Checker

Verifies that key numerical claims in the abstract, body text, and tables are internally consistent. Flags the 70 vs 71 task discrepancy, five-stage vs five-layer terminology, and validates timing claim arithmetic.

**FIX-C3 note:** The `feynman_successes` / `feynman_total` ground-truth values in Step 1 reflect the **legacy random 80/20** result (9/30) which is what the current paper text cites. Once §10.7 is updated to cite the corrected PCA 40/60 result from `exp2_pca_4060_summary.json`, those values must be updated here too. Step 6b below reads the summary file and flags any discrepancy.

## Step 1 — Load ground-truth values

In [1]:
import re
import json
import os
from pathlib import Path

TEX_FILE = "jmlr-hypatiax-paper-final.tex"
source = Path(TEX_FILE).read_text(encoding="utf-8")
lines  = source.splitlines()

# Ground-truth values from v3.0 benchmark (from tables in paper)
# FIX-C3 NOTE: feynman_successes=9 / feynman_total=30 reflects the LEGACY
# random 80/20 result cited in the current paper (§10.7).  Once §10.7 is
# revised to cite the corrected PCA 40/60 result from exp2_pca_4060_summary.json,
# update feynman_successes and feynman_total here to match.
# Step 6b below reads exp2_pca_4060_summary.json and flags any mismatch.
TRUTH = {
    "hypatix_success_pct":     89.2,
    "llm_success_pct":         62.2,
    "nn_success_pct":           5.4,
    "hypatix_gain_pp":         27.0,
    "hypatix_gain_over_nn_pp": 83.8,
    "hypatix_catastrophic":      0,
    "llm_catastrophic":          6,
    "speedup_llm_routed":       1.73,
    "llm_routed_n":             68,
    "feynman_successes":         9,   # FIX-C3: legacy 9/30 (random 80/20) — update after §10.7 revision
    "feynman_total":            30,   # FIX-C3: legacy total — update after §10.7 revision
    "nguyen_hyp":               11,
    "nguyen_pysr":              10,
    "nguyen_total":             12,
    "easy_n":                   24,
    "medium_n":                 29,
    "hard_n":                   21,
    "total_tasks":              74,
}

print("Ground-truth values loaded:")
for k, v in TRUTH.items():
    print(f"  {k:<35} = {v}")

# Resolve RESULTS_DIR for FIX-C3 checks in Step 6b
SCRIPT_DIR  = Path().resolve()
REPO_ROOT   = SCRIPT_DIR.parent if (SCRIPT_DIR.parent / "hypatiax").exists() else SCRIPT_DIR
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR",
                   str(REPO_ROOT / "hypatiax/data/results")))
print(f"\nREPO_ROOT   : {REPO_ROOT}")
print(f"RESULTS_DIR : {RESULTS_DIR}")


Ground-truth values loaded:
  hypatix_success_pct                 = 89.2
  llm_success_pct                     = 62.2
  nn_success_pct                      = 5.4
  hypatix_gain_pp                     = 27.0
  hypatix_gain_over_nn_pp             = 83.8
  hypatix_catastrophic                = 0
  llm_catastrophic                    = 6
  speedup_llm_routed                  = 1.73
  llm_routed_n                        = 68
  feynman_successes                   = 9
  feynman_total                       = 30
  nguyen_hyp                          = 11
  nguyen_pysr                         = 10
  nguyen_total                        = 12
  easy_n                              = 24
  medium_n                            = 29
  hard_n                              = 21
  total_tasks                         = 74


## Step 2 — Abstract claim verification

In [2]:
# Check abstract claims against ground truth
abstract_match = re.search(
    r'\\begin\{abstract\}(.*?)\\end\{abstract\}', source, re.DOTALL)
abstract = abstract_match.group(1) if abstract_match else ""

checks = [
    ("89.2",  "89.2% near-perfect success rate in abstract"),
    ("62.2",  "62.2% LLM baseline in abstract"),
    ("27",    "+27 pp gain in abstract"),
    ("83.8",  "+83.8 pp over NN in abstract"),
    ("1.73",  "1.73x speedup in abstract"),
    ("68",    "68/74 LLM-routed in abstract"),
    ("11/12", "Nguyen 11/12 in abstract"),
    ("9/30",  "Feynman 9/30 in abstract"),
    ("38.1",  "+38.1 pp hard tasks in abstract"),
]
print("Abstract claims verification:")
print("-" * 80)
for val, desc in checks:
    found = val in abstract
    status = "OK" if found else "MISSING"
    print(f"  [{status}]  {desc}")

Abstract claims verification:
--------------------------------------------------------------------------------
  [OK]  89.2% near-perfect success rate in abstract
  [OK]  62.2% LLM baseline in abstract
  [OK]  +27 pp gain in abstract
  [OK]  +83.8 pp over NN in abstract
  [OK]  1.73x speedup in abstract
  [OK]  68/74 LLM-routed in abstract
  [OK]  Nguyen 11/12 in abstract
  [OK]  Feynman 9/30 in abstract
  [OK]  +38.1 pp hard tasks in abstract


## Step 3 — 70 vs 71 task discrepancy (instability section)

In [3]:
# Known issue: instability section says '70 tasks' in table caption
# but '71 cases' in body text
import re

instab_section = ""
in_sec = False
for ln in lines:
    if "Stability Under Stochastic" in ln:
        in_sec = True
    if in_sec:
        instab_section += ln + "\n"
    if in_sec and r"\subsection" in ln and "Stability" not in ln:
        break

hits_70 = [(i+1, ln.strip()) for i, ln in enumerate(lines)
           if ("70 tasks" in ln or "70 cases" in ln) and i > 0]
hits_71 = [(i+1, ln.strip()) for i, ln in enumerate(lines)
           if ("71 tasks" in ln or "71 cases" in ln) and i > 0]

print("Occurrences of '70 tasks/cases':")
for lno, ctx in hits_70:
    print(f"  line {lno}: {ctx[:100]}")
print()
print("Occurrences of '71 tasks/cases':")
for lno, ctx in hits_71:
    print(f"  line {lno}: {ctx[:100]}")

print()
if hits_70 and hits_71:
    print("CONFLICT: Both '70' and '71' appear.  Decide which is correct and unify.")
    print("  Table caption (tab:instability) says '70 tasks, K=30 runs each'")
    print("  Body text says 'correlation is computed across all 71 cases'")
    print("  -> The multi-run dataset has 70 tasks (4 tasks lack 30-run data);")
    print("     Spearman correlation footnote incorrectly says 71.")
    print("  ACTION: Change '71 cases' in body text to '70 tasks'.")

Occurrences of '70 tasks/cases':
  line 1608: Table~\ref{tab:instability} summarises the regime distribution across the 70 tasks
  line 1613: \caption{LLM instability regime distribution (70 tasks, $K=30$ runs each).

Occurrences of '71 tasks/cases':
  line 532: An earlier version of this benchmark comprised 71 tasks (Easy=24, Medium=27,
  line 1637: 71 cases, with A-Symbolic cases at ($\II=0$, $\Rsq=1$) and C-Collapse cases at

CONFLICT: Both '70' and '71' appear.  Decide which is correct and unify.
  Table caption (tab:instability) says '70 tasks, K=30 runs each'
  Body text says 'correlation is computed across all 71 cases'
  -> The multi-run dataset has 70 tasks (4 tasks lack 30-run data);
     Spearman correlation footnote incorrectly says 71.
  ACTION: Change '71 cases' in body text to '70 tasks'.


## Step 4 — Five-stage vs Five-layer terminology

In [4]:
# Known issue: Abstract says 'five-stage routing' but §8 says 'Five-Layer Architecture'
hits_stage = [(i+1, ln.strip()) for i, ln in enumerate(lines)
              if "five-stage" in ln.lower() or "five stage" in ln.lower()]
hits_layer = [(i+1, ln.strip()) for i, ln in enumerate(lines)
              if "five-layer" in ln.lower() or "five layer" in ln.lower()]

print("'five-stage' occurrences:")
for lno, ctx in hits_stage:
    print(f"  line {lno}: {ctx[:100]}")
print()
print("'five-layer' occurrences:")
for lno, ctx in hits_layer:
    print(f"  line {lno}: {ctx[:100]}")
print()
if hits_stage and hits_layer:
    print("INCONSISTENCY: 'five-stage routing' vs 'Five-Layer Architecture'.")
    print("  These describe the same system but use different terminology.")
    print("  ACTION: Standardise.  The abstract/intro use 'five-stage routing'.")
    print("  Section 8.3 heading 'Five-Layer Architecture Overview' should become")
    print("  'Five-Stage Routing Architecture' to match the abstract and §7.")

'five-stage' occurrences:
  line 143: addresses this through a five-stage routing and ensembling mechanism---trust
  line 232: \item \textbf{HypatiaX hybrid system}: a five-stage routing and ensembling
  line 698: \subsection{Component 3: Five-Stage Routing and Ensembling}
  line 854: \subsection{Five-Stage Architecture Overview}
  line 987: The full five-stage routing and ensembling system described in Section~\ref{sec:routing}.
  line 1786: \item \textbf{Cascade validation rather than single-criterion acceptance.} Our five-stage routing ca
  line 1795: HypatiaX builds on three lines of prior work. AI Feynman~\citep{udrescu2020feynman} uses neural netw

'five-layer' occurrences:
  line 872: Integrates all five layers.  Achieved near-zero extrapolation error on the

INCONSISTENCY: 'five-stage routing' vs 'Five-Layer Architecture'.
  These describe the same system but use different terminology.
  ACTION: Standardise.  The abstract/intro use 'five-stage routing'.
  Section 8.3 heading 'F

## Step 5 — Timing arithmetic cross-check

In [5]:
# Verify timing numbers internally consistent
timing_claims = {
    "mean_hybrid"  : (6.8,  "HypatiaX mean 6.8s"),
    "median_hybrid": (1.7,  "HypatiaX median 1.7s"),
    "mean_nn"      : (3.0,  "Neural MLP mean 3.0s"),
    "median_nn"    : (2.7,  "Neural MLP median 2.7s"),
    "speedup"      : (1.73, "1.73x speedup LLM-routed"),
    "mean_llm"     : (11.4, "Pure LLM mean 11.4s"),
}
print("Timing numbers check (each value must appear in paper):")
for key, (val, desc) in timing_claims.items():
    val_str = str(val)
    found = val_str in source
    status = "OK" if found else "MISSING"
    print(f"  [{status}]  {desc}")

print()
# Cross-check: 1.73x speedup from median NN 2.7s
estimated_speedup = 2.7 / 1.56  # median NN / estimated LLM-routed median
print(f"  Cross-check: NN median 2.7s / 1.73 = {2.7/1.73:.2f}s (should be ~1.56s for LLM-routed)")
print("  App C says 1.73x from 2.7s NN median -> LLM-routed median ~1.56s (marked with *)")

Timing numbers check (each value must appear in paper):
  [OK]  HypatiaX mean 6.8s
  [OK]  HypatiaX median 1.7s
  [OK]  Neural MLP mean 3.0s
  [OK]  Neural MLP median 2.7s
  [OK]  1.73x speedup LLM-routed
  [OK]  Pure LLM mean 11.4s

  Cross-check: NN median 2.7s / 1.73 = 1.56s (should be ~1.56s for LLM-routed)
  App C says 1.73x from 2.7s NN median -> LLM-routed median ~1.56s (marked with *)


## Step 6 — Fix recipe

## Step 6b — FIX-C3: Feynman corrected result vs paper claim

Reads `exp2_pca_4060_summary.json` (written by `run_all.sh exp2_feynman_pca_4060` after C5c-0 completes) and compares the corrected PCA 40/60 solve rate against the legacy 9/30 value in `TRUTH` above. Flags if the paper text still cites the legacy number after §10.7 has been revised.

In [ ]:
# FIX-C3: compare corrected Feynman result against TRUTH dict
FEYNMAN_SUMMARY = (
    RESULTS_DIR
    / "comparison_results/feynman-tests/exp2_pca_4060/exp2_pca_4060_summary.json"
)
BASELINE_FILE = RESULTS_DIR / "fixc3_baseline.json"

print("FIX-C3: Feynman corrected result check")
print("-" * 80)

if not FEYNMAN_SUMMARY.exists():
    print(f"  [SKIP] {FEYNMAN_SUMMARY.name} not found")
    print("  C5c-0 (exp2_feynman_pca_4060) has not completed yet — skipping.")
else:
    data = json.loads(FEYNMAN_SUMMARY.read_text())
    corrected_n_pass  = data.get("n_pass",  None)
    corrected_n_total = data.get("n_total", None)
    corrected_rate    = data.get("solve_rate", None)
    protocol          = data.get("split_protocol", "?")

    legacy_n_pass  = TRUTH["feynman_successes"]
    legacy_n_total = TRUTH["feynman_total"]

    print(f"  Legacy  (random 80/20, TRUTH dict) : {legacy_n_pass}/{legacy_n_total} "
          f"= {legacy_n_pass/legacy_n_total:.3f}")

    if corrected_rate is None:
        print(f"  [FAIL] corrected solve_rate is null — C5c-0 domains may not have finished")
    else:
        print(f"  Corrected (PCA 40/60, {protocol}) : "
              f"{corrected_n_pass}/{corrected_n_total} = {corrected_rate:.3f}")

        if corrected_n_pass == legacy_n_pass and corrected_n_total == legacy_n_total:
            print("  [WARN] Corrected result equals legacy result — "
                  "verify pca_directed_split was actually applied (not a checkpoint replay)")
        elif corrected_rate < legacy_n_pass / legacy_n_total:
            print("  [OK]  Corrected rate LOWER than legacy — expected for harder PCA 40/60 split")
        else:
            print("  [WARN] Corrected rate HIGHER than legacy — unexpected; verify split was applied")

        # Check whether paper text still cites legacy value
        legacy_str    = f"{legacy_n_pass}/{legacy_n_total}"
        corrected_str = f"{corrected_n_pass}/{corrected_n_total}"
        legacy_in_paper    = legacy_str in source
        corrected_in_paper = corrected_str in source

        print()
        print(f"  Paper cites legacy {legacy_str!r}    : {legacy_in_paper}")
        print(f"  Paper cites corrected {corrected_str!r} : {corrected_in_paper}")

        if legacy_in_paper and not corrected_in_paper:
            print()
            print("  [ACTION REQUIRED] Paper still cites the legacy random 80/20 result.")
            print("  Update §10.7 to cite the corrected PCA 40/60 result, or add a")
            print("  disclosure note explaining both numbers. Then update TRUTH dict:")
            print(f"    feynman_successes = {corrected_n_pass}")
            print(f"    feynman_total     = {corrected_n_total}")
        elif corrected_in_paper:
            print("  [OK]  Paper cites corrected result — TRUTH dict should be updated to match")
            if TRUTH['feynman_successes'] != corrected_n_pass or TRUTH['feynman_total'] != corrected_n_total:
                print(f"  [ACTION] Update TRUTH dict: feynman_successes={corrected_n_pass}, "
                      f"feynman_total={corrected_n_total}")

# Baseline file check
if BASELINE_FILE.exists():
    bl = json.loads(BASELINE_FILE.read_text())
    print(f"\n  [OK]  fixc3_baseline.json: {bl.get('n_pass')}/{bl.get('n_total')} "
          f"(rate={bl.get('solve_rate')}, claim={bl.get('paper_claim')})")
else:
    print("\n  [INFO] fixc3_baseline.json not present — Gate C has not run yet")


In [6]:
fixes = (
    "FIX-N1  '71 cases' should be '70 tasks' in instability section body text.\n"
    "  Table caption (tab:instability) correctly says '70 tasks'.\n"
    "  Spearman footnote incorrectly says '71 cases' -> change to '70 tasks'.\n\n"
    "FIX-N2  Terminology: 'five-stage routing' vs 'Five-Layer Architecture'\n"
    "  Abstract, intro, and §7 say 'five-stage routing and ensembling'.\n"
    "  §8.3 heading says 'Five-Layer Architecture Overview'.\n"
    "  -> Rename §8.3 to 'Five-Stage Architecture Overview' for consistency.\n\n"
    "FIX-N3  Verify abstract numbers after Nguyen-12 seed=123 rerun:\n"
    "  If hv50-2 seed=123 changes PySR count (currently 10/12), update:\n"
    "  abstract '10/12 for PySR-only', §10.8 Table 7, MW statistics.\n\n"
    "FIX-C3  Feynman split protocol mismatch (§10.7):\n"
    "  Legacy 9/30 used random 80/20 split; corrected result uses PCA 40/60.\n"
    "  See Step 6b above for the corrected solve rate from exp2_pca_4060_summary.json.\n"
    "  ACTION: Update §10.7 to cite corrected result, then update TRUTH dict:\n"
    "    feynman_successes = <corrected n_pass from Step 6b>\n"
    "    feynman_total     = <corrected n_total from Step 6b>\n"
    "  Also update abstract '9/30' claim once §10.7 is revised.\n"
)
print(fixes)


FIX-N1  '71 cases' should be '70 tasks' in instability section body text.
  Table caption (tab:instability) correctly says '70 tasks'.
  Spearman footnote incorrectly says '71 cases' -> change to '70 tasks'.

FIX-N2  Terminology: 'five-stage routing' vs 'Five-Layer Architecture'
  Abstract, intro, and §7 say 'five-stage routing and ensembling'.
  §8.3 heading says 'Five-Layer Architecture Overview'.
  -> Rename §8.3 to 'Five-Stage Architecture Overview' for consistency.

FIX-N3  Verify abstract numbers after Nguyen-12 seed=123 rerun:
  If hv50-2 seed=123 changes PySR count (currently 10/12), update:
  abstract '10/12 for PySR-only', §10.8 Table 7, MW statistics.

